# OpenSanctions bundle - London subset

## Set up

In [1]:
from dataclasses import dataclass, field
import json
import math
import os
import pathlib
import sys
import typing

from graphdatascience import GraphDataScience
from icecream import ic
from tqdm import tqdm
import kuzu
import dotenv
import neo4j
import numpy as np
import pandas as pd
import watermark

%load_ext watermark

In [2]:
%watermark
%watermark --iversions

Last updated: 2024-08-13T19:10:32.300764-07:00

Python implementation: CPython
Python version       : 3.11.9
IPython version      : 8.26.0

Compiler    : Clang 13.0.0 (clang-1300.0.29.30)
OS          : Darwin
Release     : 23.5.0
Machine     : arm64
Processor   : arm
CPU cores   : 14
Architecture: 64bit

numpy    : 1.26.4
sys      : 3.11.9 (v3.11.9:de54cf5be3, Apr  2 2024, 07:12:50) [Clang 13.0.0 (clang-1300.0.29.30)]
neo4j    : 5.23.1
watermark: 2.4.3
kuzu     : 0.5.0
json     : 2.0.9
pandas   : 2.2.2



## Connect to Neo4j

Establish a GDS connection to Neo4j.

In [3]:
dotenv.load_dotenv(dotenv.find_dotenv())

bolt_uri: str = os.environ.get("NEO4J_BOLT")
database: str = os.environ.get("NEO4J_DBMS")
username: str = os.environ.get("NEO4J_USER")
password: str = os.environ.get("NEO4J_PASS")

gds:GraphDataScience = GraphDataScience(
    bolt_uri,
    auth = ( username, password, ),
    database = database,
    aura_ds = False,
)

Define a function to load "chunked" Pandas dataframes into Neo4j using mini-batch.

In [4]:
MAX_ROWS: int = 25000

def load_neo4j_df (
    df: pd.DataFrame,
    query: str,
    ) -> None:
    n_splits: int = math.ceil(len(df) / MAX_ROWS)

    for df_chunk in tqdm(np.array_split(df, n_splits), desc = "chunks"):
        gds.run_cypher(
            query,
            {"rows": df_chunk.to_dict(orient = "records")},
        )

In [5]:
gds.run_cypher("""
DROP CONSTRAINT `org_node_key` IF EXISTS
""")

gds.run_cypher("""
CREATE CONSTRAINT `org_node_key` IF NOT EXISTS
  FOR (l:Organization)
  REQUIRE l.record_id IS NODE KEY
""")

gds.run_cypher("""
DROP INDEX `org_node_id` IF EXISTS
""")

gds.run_cypher("""
CREATE INDEX org_node_id IF NOT EXISTS
  FOR (l:Organization) ON (l.record_id)
""")

""


In [6]:
gds.run_cypher("""
DROP CONSTRAINT `entity_node_key` IF EXISTS
""")

gds.run_cypher("""
CREATE CONSTRAINT `entity_node_key` IF NOT EXISTS
  FOR (l:Entity)
  REQUIRE l.uid IS NODE KEY
""")

gds.run_cypher("""
DROP INDEX `entity_node_id` IF EXISTS
""")

gds.run_cypher("""
CREATE INDEX entity_node_id IF NOT EXISTS
  FOR (l:Entity) ON (l.uid)
""")

""


## Data discovery

In [7]:
dat_dir: pathlib.Path = pathlib.Path("subset")
os_id_set: typing.Set[ str ] = set()

In [8]:
def get_property_keys (
    df: pd.DataFrame,
    ) -> typing.List[ str ]:
    """
Convert column names from the given Pandas dataframe into Cypher property names.
    """
    return [
        name.lower().replace(" ", "_")
        for name in df.columns.values.tolist()
    ]

### Sanctions list

load file: `sanctions.json` for 
> Government-published sanctions lists

In [9]:
dat_file: pathlib.Path = dat_dir / "sanctions.json"

with dat_file.open("r", encoding = "utf-8") as fp:
    df: pd.DataFrame = pd.DataFrame([
        json.loads(line)
        for line in tqdm(fp.readlines(), desc = "read JSON")
    ]).astype(str).fillna("")

df.columns = get_property_keys(df)
df.replace({"nan": ""}, regex = True, inplace = True)
df.head(3)

read JSON: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 306/306 [00:00<00:00, 84560.35it/s]


,data_source,record_id,record_type,last_change,names,risks,addresses,countries,identifiers,source_links,relationships,url,gender,dates,contacts
0,OPEN_SANCTIONS,NK-276gB7L8sFEhctkuEpZsus,ORGANIZATION,2024-05-07T15:53:01,"[{'NAME_TYPE': 'PRIMARY', 'NAME_ORG': 'BANK SA...",[{'TOPIC': 'sanction'}],"[{'ADDR_FULL': 'PO Box 15175/584, 6th Floor, S...","[{'CITIZENSHIP': 'gb'}, {'CITIZENSHIP': 'ir'}]","[{'NATIONAL_ID_NUMBER': '01126618'}, {'OTHER_I...",[{'SOURCE_URL': 'https://sanctionssearch.ofac....,"[{'REL_POINTER_ROLE': 'related-to', 'REL_POINT...",https://www.opensanctions.org/entities/NK-276g...,,,
1,OPEN_SANCTIONS,NK-2RoX4fFRx9FtQE9osMzwk9,ORGANIZATION,2024-03-06T18:31:06,"[{'NAME_TYPE': 'PRIMARY', 'NAME_ORG': 'ElEi Ho...",[{'TOPIC': 'sanction'}],[{'ADDR_FULL': 'Сполучене Королівство Великої ...,[{'REGISTRATION_COUNTRY': 'io'}],"[{'NATIONAL_ID_NUMBER': '13552425'}, {'OTHER_I...",,,https://www.opensanctions.org/entities/NK-2RoX...,,,
2,OPEN_SANCTIONS,NK-3KRoVfPHitLoVonaD9y6bE,PERSON,2024-05-09T21:39:15,"[{'NAME_TYPE': 'PRIMARY', 'NAME_FULL': 'Ashraf...",[{'TOPIC': 'sanction'}],"[{'ADDR_FULL': '1 College Yard, Winchester Ave...","[{'NATIONALITY': 'sd'}, {'NATIONALITY': 'ss'},...","[{'PASSPORT_NUMBER': 'B00018325'}, {'NATIONAL_...",[{'SOURCE_URL': 'https://sanctionssearch.ofac....,[{'REL_POINTER_ROLE': 'Owned or Controlled By'...,https://www.opensanctions.org/entities/NK-3KRo...,M,"[{'DATE_OF_BIRTH': '1957-01-31'}, {'DATE_OF_BI...",


In [10]:
df.describe(include = "all").loc[[ "count", "freq", "unique", "top", ]]

,data_source,record_id,record_type,last_change,names,risks,addresses,countries,identifiers,source_links,relationships,url,gender,dates,contacts
count,306,306,306,306,306,306,306,306,306,306,306,306,306,306,306
freq,306,1,170,109,10,151,149,202,1,192,246,1,263,242,293
unique,1,306,3,34,271,5,145,68,306,115,58,306,3,65,14
top,OPEN_SANCTIONS,NK-276gB7L8sFEhctkuEpZsus,None,2023-11-02T16:38:16,"[{'NAME_TYPE': 'PRIMARY', 'NAME_FULL': 'London'}]",,,[{'CITIZENSHIP': 'gb'}],"[{'NATIONAL_ID_NUMBER': '01126618'}, {'OTHER_I...",,,https://www.opensanctions.org/entities/NK-276g...,,,


In [11]:
os_id_set |= set(df.record_id.values)
len(os_id_set)

306

In [12]:
sanction_ids: typing.Set[ str ] = set(df.record_id.values)

Load records into Neo4j

In [13]:
list(df.columns)

['data_source',
 'record_id',
 'record_type',
 'last_change',
 'names',
 'risks',
 'addresses',
 'countries',
 'identifiers',
 'source_links',
 'relationships',
 'url',
 'gender',
 'dates',
 'contacts']

In [14]:
unwind_query: str = """
UNWIND $rows AS row
CALL {
  WITH row
  MERGE (n:Organization {record_id: row.record_id})
  SET n += {
    data_source: row.data_source,
    record_type: row.record_type,
    last_change: row.last_change,
    names: row.names,
    risks: row.risks,
    addresses: row.addresses,
    countries: row.countries,
    identifiers: row.identifiers,
    source_links: row.source_links,
    relationships: row.relationships,
    url: row.url,
    gender: row.gender,
    dates: row.dates,
    contacts: row.contacts
  }
} IN TRANSACTIONS OF 5000 ROWS
    """

load_neo4j_df(df, unwind_query)

/Users/paco/src/ERKG/2_london/opensanctions/venv/lib/python3.11/site-packages/numpy/core/fromnumeric.py:59: FutureWarning: 'DataFrame.swapaxes' is deprecated and will be removed in a future version. Please use 'DataFrame.transpose' instead.
  return bound(*args, **kwds)
chunks: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00, 21.00it/s]


### Default set

load file: `default.json` for 
> Full sanctions, PEP, crime and associated risk graph

In [15]:
dat_file: pathlib.Path = dat_dir / "default.json"

with dat_file.open("r", encoding = "utf-8") as fp:
    df: pd.DataFrame = pd.DataFrame([
        json.loads(line)
        for line in tqdm(fp.readlines(), desc = "read JSON")
    ]).astype(str).fillna("")

df.columns = get_property_keys(df)
df.replace({"nan": ""}, regex = True, inplace = True)
df.head(3)

read JSON: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████| 7183/7183 [00:00<00:00, 121625.81it/s]


,data_source,record_id,record_type,last_change,names,risks,addresses,dates,countries,contacts,identifiers,source_links,relationships,url,gender
0,OPEN_SANCTIONS,NK-276gB7L8sFEhctkuEpZsus,ORGANIZATION,2024-08-07T10:22:04,"[{'NAME_TYPE': 'PRIMARY', 'NAME_ORG': 'BANK SA...","[{'TOPIC': 'sanction'}, {'TOPIC': 'fin.bank'}]","[{'ADDR_FULL': '120 MOORGATE, LONDON EC2M 6TS,...",[{'REGISTRATION_DATE': '1973-08-03'}],"[{'CITIZENSHIP': 'ir'}, {'CITIZENSHIP': 'gb'},...",[{'WEBSITE_ADDRESS': 'https://www.saderat-plc....,"[{'NATIONAL_ID_NUMBER': '01126618'}, {'NATIONA...",[{'SOURCE_URL': 'https://permid.org/1-50008660...,"[{'REL_POINTER_ROLE': 'related-to', 'REL_POINT...",https://www.opensanctions.org/entities/NK-276g...,
1,OPEN_SANCTIONS,NK-2RoX4fFRx9FtQE9osMzwk9,ORGANIZATION,2024-03-06T18:31:06,"[{'NAME_TYPE': 'PRIMARY', 'NAME_ORG': 'ElEi Ho...",[{'TOPIC': 'sanction'}],"[{'ADDR_FULL': '61A Asplins Road, London, Unit...",,[{'REGISTRATION_COUNTRY': 'io'}],,"[{'NATIONAL_ID_NUMBER': '13552425'}, {'OTHER_I...",,,https://www.opensanctions.org/entities/NK-2RoX...,
2,OPEN_SANCTIONS,NK-2WqqzAmKXFyrgf7xMDjYMv,PERSON,2024-06-07T08:01:59,"[{'NAME_TYPE': 'PRIMARY', 'NAME_FULL': 'SHERRI...",[{'TOPIC': 'debarment'}],"[{'ADDR_FULL': 'LONDON, ONTARIO, CANADA, TX'},...",[{'DATE_OF_BIRTH': '1970-03-23'}],[{'CITIZENSHIP': 'us'}],,"[{'OTHER_ID_TYPE': 'OPEN_SANCTIONS', 'OTHER_ID...",,,https://www.opensanctions.org/entities/NK-2Wqq...,


In [16]:
df.describe(include = "all").loc[[ "count", "freq", "unique", "top", ]]

,data_source,record_id,record_type,last_change,names,risks,addresses,dates,countries,contacts,identifiers,source_links,relationships,url,gender
count,7183,7183,7183,7183,7183,7183,7183,7183,7183,7183,7183,7183,7183,7183,7183
freq,7183,1,5456,5177,34,4891,376,5681,5386,6976,1,6104,5598,1,6683
unique,1,7183,3,262,6169,51,5444,1438,263,206,7183,1079,1468,7183,3
top,OPEN_SANCTIONS,NK-276gB7L8sFEhctkuEpZsus,ORGANIZATION,2023-03-16T00:00:00,"[{'NAME_TYPE': 'PRIMARY', 'NAME_ORG': 'Organiz...",[{'TOPIC': 'fin.bank'}],[{'PLACE_OF_BIRTH': 'London'}],,[{'CITIZENSHIP': 'gb'}],,"[{'NATIONAL_ID_NUMBER': '01126618'}, {'NATIONA...",,,https://www.opensanctions.org/entities/NK-276g...,


In [17]:
os_id_set |= set(df.record_id.values)
len(os_id_set)

7183

Note that the `record_id` values for `default.json` include all of those from `sanctions.json` as well. So we'll filter out these records.

In [18]:
df = df[df["record_id"].isin(list(os_id_set.difference(sanction_ids)))]
df.head(3)

,data_source,record_id,record_type,last_change,names,risks,addresses,dates,countries,contacts,identifiers,source_links,relationships,url,gender
2,OPEN_SANCTIONS,NK-2WqqzAmKXFyrgf7xMDjYMv,PERSON,2024-06-07T08:01:59,"[{'NAME_TYPE': 'PRIMARY', 'NAME_FULL': 'SHERRI...",[{'TOPIC': 'debarment'}],"[{'ADDR_FULL': 'LONDON, ONTARIO, CANADA, TX'},...",[{'DATE_OF_BIRTH': '1970-03-23'}],[{'CITIZENSHIP': 'us'}],,"[{'OTHER_ID_TYPE': 'OPEN_SANCTIONS', 'OTHER_ID...",,,https://www.opensanctions.org/entities/NK-2Wqq...,
3,OPEN_SANCTIONS,NK-2uLiev7s4AV8Rkw2bhtbGj,ORGANIZATION,2023-03-16T00:00:00,"[{'NAME_TYPE': 'PRIMARY', 'NAME_ORG': 'GP BULL...",[{'TOPIC': 'fin.bank'}],[{'ADDR_FULL': 'CHISWICK 52 JERMYN STREET LOND...,,[{'CITIZENSHIP': 'gb'}],,"[{'OTHER_ID_TYPE': 'swiftBic', 'OTHER_ID_NUMBE...",,,https://www.opensanctions.org/entities/NK-2uLi...,
5,OPEN_SANCTIONS,NK-3NxtmBPoRPRiji5JKHtzTT,ORGANIZATION,2023-03-16T00:00:00,"[{'NAME_TYPE': 'PRIMARY', 'NAME_ORG': 'TRANZFA...",,[{'ADDR_FULL': 'CANADA SQUARE ONE CANADA SQUAR...,,[{'CITIZENSHIP': 'gb'}],,"[{'OTHER_ID_TYPE': 'swiftBic', 'OTHER_ID_NUMBE...",,,https://www.opensanctions.org/entities/NK-3Nxt...,


In [19]:
df.describe(include = "all").loc[[ "count", "freq", "unique", "top", ]]

,data_source,record_id,record_type,last_change,names,risks,addresses,dates,countries,contacts,identifiers,source_links,relationships,url,gender
count,6877,6877,6877,6877,6877,6877,6877,6877,6877,6877,6877,6877,6877,6877,6877
freq,6877,1,5374,5177,34,4891,376,5469,5206,6695,1,5921,5373,1,6422
unique,1,6877,3,228,5896,39,5296,1345,200,181,6877,956,1388,6877,3
top,OPEN_SANCTIONS,NK-2WqqzAmKXFyrgf7xMDjYMv,ORGANIZATION,2023-03-16T00:00:00,"[{'NAME_TYPE': 'PRIMARY', 'NAME_ORG': 'Organiz...",[{'TOPIC': 'fin.bank'}],[{'PLACE_OF_BIRTH': 'London'}],,[{'CITIZENSHIP': 'gb'}],,"[{'OTHER_ID_TYPE': 'OPEN_SANCTIONS', 'OTHER_ID...",,,https://www.opensanctions.org/entities/NK-2Wqq...,


Load records into Neo4j

In [20]:
list(df.columns)

['data_source',
 'record_id',
 'record_type',
 'last_change',
 'names',
 'risks',
 'addresses',
 'dates',
 'countries',
 'contacts',
 'identifiers',
 'source_links',
 'relationships',
 'url',
 'gender']

In [21]:
unwind_query: str = """
UNWIND $rows AS row
CALL {
  WITH row
  MERGE (n:Organization {record_id: row.record_id})
  SET n += {
    data_source: row.data_source,
    record_type: row.record_type,
    last_change: row.last_change,
    names: row.names,
    risks: row.risks,
    addresses: row.addresses,
    countries: row.countries,
    identifiers: row.identifiers,
    source_links: row.source_links,
    relationships: row.relationships,
    url: row.url,
    gender: row.gender,
    dates: row.dates,
    contacts: row.contacts
  }
} IN TRANSACTIONS OF 5000 ROWS
    """

load_neo4j_df(df, unwind_query)

/Users/paco/src/ERKG/2_london/opensanctions/venv/lib/python3.11/site-packages/numpy/core/fromnumeric.py:59: FutureWarning: 'DataFrame.swapaxes' is deprecated and will be removed in a future version. Please use 'DataFrame.transpose' instead.
  return bound(*args, **kwds)
chunks: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00,  2.59it/s]


### GLEIF identifiers

load file: `gleif.json` for 
> Companies that have a legal entity identifier (LEI)

In [22]:
dat_file: pathlib.Path = dat_dir / "gleif.json"

with dat_file.open("r", encoding = "utf-8") as fp:
    df: pd.DataFrame = pd.DataFrame([
        json.loads(line)
        for line in tqdm(fp.readlines(), desc = "read JSON")
    ]).astype(str).fillna("")

df.columns = get_property_keys(df)
df.replace({"nan": ""}, regex = True, inplace = True)
df.head(3)

read JSON: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████| 63145/63145 [00:00<00:00, 135611.05it/s]


,data_source,record_id,record_type,last_change,names,addresses,dates,countries,identifiers,url,relationships,risks
0,OS_GLEIF,lei-03EINY24LQ6IXW124R72,ORGANIZATION,2024-07-10T08:08:08,"[{'NAME_TYPE': 'PRIMARY', 'NAME_ORG': 'THE BAR...",[{'ADDR_FULL': 'C/O BARCLAYS PENSION FUNDS TRU...,[{'REGISTRATION_DATE': '2012-09-24'}],[{'REGISTRATION_COUNTRY': 'gb'}],"[{'LEI_NUMBER': '03EINY24LQ6IXW124R72'}, {'OTH...",https://www.opensanctions.org/entities/lei-03E...,,
1,OS_GLEIF,lei-05MQKGBWLLX7RPPDO189,ORGANIZATION,2024-06-12T08:08:21,"[{'NAME_TYPE': 'PRIMARY', 'NAME_ORG': 'THE MAR...","[{'ADDR_FULL': '16 HATFIELDS, LONDON SE1 8DJ, ...",,[{'REGISTRATION_COUNTRY': 'gb'}],"[{'NATIONAL_ID_NUMBER': '03909510'}, {'LEI_NUM...",https://www.opensanctions.org/entities/lei-05M...,,
2,OS_GLEIF,lei-0677F7L8DTFEBS5RWE15,ORGANIZATION,2024-06-12T08:08:21,"[{'NAME_TYPE': 'PRIMARY', 'NAME_ORG': 'CARRING...","[{'ADDR_FULL': '5 CANADA SQUARE, LONDON E14 5A...",,[{'REGISTRATION_COUNTRY': 'gb'}],"[{'NATIONAL_ID_NUMBER': '06345337'}, {'LEI_NUM...",https://www.opensanctions.org/entities/lei-067...,,


In [23]:
df.describe(include = "all").loc[[ "count", "freq", "unique", "top", ]]

,data_source,record_id,record_type,last_change,names,addresses,dates,countries,identifiers,url,relationships,risks
count,63145,63145,63145,63145,63145,63145,63145,63145,63145,63145,63145,63145
freq,63145,1,63145,42170,4,440,24802,57099,1,1,49182,60903
unique,1,63145,1,56,63007,35299,10987,99,63145,63145,13962,2
top,OS_GLEIF,lei-03EINY24LQ6IXW124R72,ORGANIZATION,2024-06-12T08:08:21,"[{'NAME_TYPE': 'PRIMARY', 'NAME_ORG': 'BLACKRO...",[{'ADDR_FULL': 'C/O HSBC TRUST COMPANY (UK) LI...,,[{'REGISTRATION_COUNTRY': 'gb'}],"[{'LEI_NUMBER': '03EINY24LQ6IXW124R72'}, {'OTH...",https://www.opensanctions.org/entities/lei-03E...,,


In [24]:
os_id_set |= set(df.record_id.values)
len(os_id_set)

70317

Load records into Neo4j

In [25]:
list(df.columns)

['data_source',
 'record_id',
 'record_type',
 'last_change',
 'names',
 'addresses',
 'dates',
 'countries',
 'identifiers',
 'url',
 'relationships',
 'risks']

In [26]:
unwind_query: str = """
UNWIND $rows AS row
CALL {
  WITH row
  MERGE (n:Organization {record_id: row.record_id})
  SET n += {
    data_source: row.data_source,
    record_type: row.record_type,
    last_change: row.last_change,
    names: row.names,
    risks: row.risks,
    addresses: row.addresses,
    countries: row.countries,
    identifiers: row.identifiers,
    relationships: row.relationships,
    url: row.url,
    dates: row.dates
  }
} IN TRANSACTIONS OF 5000 ROWS
    """

load_neo4j_df(df, unwind_query)

/Users/paco/src/ERKG/2_london/opensanctions/venv/lib/python3.11/site-packages/numpy/core/fromnumeric.py:59: FutureWarning: 'DataFrame.swapaxes' is deprecated and will be removed in a future version. Please use 'DataFrame.transpose' instead.
  return bound(*args, **kwds)
chunks: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 3/3 [00:02<00:00,  1.18it/s]


## Parse the results from Senzing

Let's define a `dataclass` to represent the parsed results from Senzing entity resolution.

In [27]:
@dataclass(order=False, frozen=False)
class Entity:  # pylint: disable=R0902
    """
A data class representing a resolved entity.
    """
    entity_uid: str
    name: str
    num_recs: int
    records: typing.Dict[ str, str ] = field(default_factory = lambda: {})
    related: typing.Dict[ str, dict ] = field(default_factory = lambda: {})
    has_ref: bool = False

Parse the JSON data from the export, to build a dictionary of entities indexed by their unique identifiers. Also keep track of both the "resolved" and "related" records for each entity, to use for constructing the knowledge graph from these results.

In [28]:
export_path: pathlib.Path = pathlib.Path("export.json")
entities: dict = {}

with export_path.open() as fp:
    for line in tqdm(fp.readlines(), desc = "read JSON"):
        entity_dat: dict = json.loads(line)
        entity_uid: str = str(entity_dat["RESOLVED_ENTITY"]["ENTITY_ID"])

        entity_name: str = ""
        records: dict = {}

        for rec in entity_dat["RESOLVED_ENTITY"]["RECORDS"]:
            record_uid: str = str(rec["RECORD_ID"])
            match_key: str = rec["MATCH_KEY"]

            if match_key.strip() == "":
                match_key = "INITIAL"
            records[record_uid] = match_key

            if entity_name == "" and rec["ENTITY_DESC"] != "":
                entity_name = rec["ENTITY_DESC"]

        if entity_name == "":
            entity_name = entity_uid

        entities[entity_uid] = Entity(
            entity_uid = entity_uid,
            name = entity_name,
            records = records,
            num_recs = len(records),
            related = {
                str(r["ENTITY_ID"]): r
                for r in entity_dat["RELATED_ENTITIES"]
            },
        )

read JSON: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████| 68608/68608 [00:00<00:00, 86948.81it/s]


To finish preparing the input data for resolved entities, let's make a quick traversal of the record linkage and set a flag for "interesting" entities which will have relations in the graph to visualize.

In [29]:
for entity in entities.values():
    if entity.num_recs > 0:
        entity.has_ref = True

    for rel_ent_id in entity.related:
        entities[rel_ent_id].has_ref = True

Let's examine one of the resolved entity objects, to see which fields are available.

In [30]:
entity_dat

{'RESOLVED_ENTITY': {'ENTITY_ID': 300144,
  'RECORDS': [{'DATA_SOURCE': 'OPEN_SANCTIONS',
    'RECORD_ID': 'trade-csl-cc60ebd4d79904e74fe9e05989f0e026861f6b547ad8b13ba69812a3',
    'ENTITY_TYPE': 'GENERIC',
    'INTERNAL_ID': 300144,
    'ENTITY_KEY': 'CB5295DEB9156BF57BB0939C888674F07910D144',
    'ENTITY_DESC': 'Sergei (Sergi) Ivanovich Tomchani',
    'MATCH_KEY': '',
    'MATCH_LEVEL': 0,
    'MATCH_LEVEL_CODE': '',
    'ERRULE_CODE': '',
    'LAST_SEEN_DT': '2024-08-13 21:41:59.274'}]},
 'RELATED_ENTITIES': [{'ENTITY_ID': 3923,
   'MATCH_LEVEL': 3,
   'MATCH_LEVEL_CODE': 'POSSIBLY_RELATED',
   'MATCH_KEY': '+ADDRESS',
   'ERRULE_CODE': 'SFF',
   'IS_DISCLOSED': 0,
   'IS_AMBIGUOUS': 0,
   'RECORDS': [{'DATA_SOURCE': 'OS_GLEIF',
     'RECORD_ID': 'lei-2138001DC844QGM9SH02'}]},
  {'ENTITY_ID': 23587,
   'MATCH_LEVEL': 3,
   'MATCH_LEVEL_CODE': 'POSSIBLY_RELATED',
   'MATCH_KEY': '+ADDRESS',
   'ERRULE_CODE': 'CFF',
   'IS_DISCLOSED': 0,
   'IS_AMBIGUOUS': 0,
   'RECORDS': [{'DATA_SOU

Load the Senzing overlay into Neo4j

In [31]:
df: pd.DataFrame = pd.DataFrame([
    {
        "uid": entity.entity_uid,
        "name": entity.name,
        "has_ref": entity.has_ref,
    }
    for entity in entities.values()
])

df.head()

,uid,name,has_ref
0,1,C H. PETT WILL TRUST - RESIDUARY FUND,True
1,2,PACIFIC QUAY FINANCE PLC,True
2,3,Freremon Limited,True
3,4,PARITY BIDCO LIMITED,True
4,5,"PRAESIDIAN CAPITAL EUROPE I-B, LP",True


In [32]:
df.describe(include = "all").loc[[ "count", "freq", "unique", "top", ]]

,uid,name,has_ref
count,68608,68608,68608
freq,1,31,68608
unique,68608,67585,1
top,1,Organization,True


In [33]:
unwind_query: str = """
UNWIND $rows AS row
CALL {
  WITH row
  MERGE (n:Entity {uid: row.uid})
  SET n += {
    name: row.name,
    has_ref: row.has_ref
  }
} IN TRANSACTIONS OF 5000 ROWS
    """

load_neo4j_df(df, unwind_query)

/Users/paco/src/ERKG/2_london/opensanctions/venv/lib/python3.11/site-packages/numpy/core/fromnumeric.py:59: FutureWarning: 'DataFrame.swapaxes' is deprecated and will be removed in a future version. Please use 'DataFrame.transpose' instead.
  return bound(*args, **kwds)
chunks: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 3/3 [00:00<00:00,  3.64it/s]


Now we'll connect each entity with its resolved records.

In [34]:
df: pd.DataFrame = pd.DataFrame([
    {
        "entity_uid": str(entity.entity_uid),
        "record_uid": record_uid,
        "match_key": match_key,
    }
    for entity in entities.values()
    for record_uid, match_key in entity.records.items()
])

df.head()

,entity_uid,record_uid,match_key
0,1,lei-213800HI8LJWKODMEM62,INITIAL
1,2,lei-549300FKFB3A6S6C6H29,INITIAL
2,3,lei-254900WFRYQ62G0VW554,INITIAL
3,4,lei-2549005UZ6D5ZW1JWO94,INITIAL
4,5,lei-549300TEEMM5FNYNVJ33,INITIAL


In [35]:
df.describe(include = "all").loc[[ "count", "freq", "unique", "top", ]]

,entity_uid,record_uid,match_key
count,70317,70317,70317
freq,17,1,68599
unique,68608,70317,31
top,41361,lei-213800HI8LJWKODMEM62,INITIAL


In [36]:
unwind_query: str = """
UNWIND $rows AS row
CALL {
  WITH row
  MATCH
    (ent:Entity {uid: row.entity_uid}),
    (org:Organization {record_id: row.record_uid})       
  MERGE (ent)-[:RESOLVES {match_key: row.match_key}]->(org)
} IN TRANSACTIONS OF 5000 ROWS
    """

load_neo4j_df(df, unwind_query)

/Users/paco/src/ERKG/2_london/opensanctions/venv/lib/python3.11/site-packages/numpy/core/fromnumeric.py:59: FutureWarning: 'DataFrame.swapaxes' is deprecated and will be removed in a future version. Please use 'DataFrame.transpose' instead.
  return bound(*args, **kwds)
chunks: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 3/3 [00:01<00:00,  2.41it/s]


Similarly, we’ll connect each entity with its related entities, if any.

In [37]:
df: pd.DataFrame = pd.DataFrame([
    {
        "entity_uid": str(entity.entity_uid),
        "rel_ent": str(rel_ent["ENTITY_ID"]),
        "ambiguous": (rel_ent["IS_AMBIGUOUS"] == 0),
        "disclosed": (rel_ent["IS_DISCLOSED"] == 0),
        "match_level": rel_ent["MATCH_LEVEL"],
        "match_level_code": rel_ent["MATCH_LEVEL_CODE"],
    }
    for entity in entities.values()
    for rel_key, rel_ent in entity.related.items()
])

df.head(3)

,entity_uid,rel_ent,ambiguous,disclosed,match_level,match_level_code
0,1,100660,True,True,3,POSSIBLY_RELATED
1,4,10435,True,True,3,POSSIBLY_RELATED
2,4,13391,True,True,3,POSSIBLY_RELATED


In [38]:
unwind_query: str = """
UNWIND $rows AS row
CALL {
  WITH row
  MATCH
    (ent:Entity {uid: row.entity_uid}),
    (rel_ent:Entity {uid: row.rel_ent})       
  MERGE (ent)-[:RELATED {ambiguous: row.ambiguous, disclosed: row.disclosed, match_level: row.match_level, match_level_code: row.match_level_code}]->(rel_ent)
} IN TRANSACTIONS OF 5000 ROWS
    """

load_neo4j_df(df, unwind_query)

/Users/paco/src/ERKG/2_london/opensanctions/venv/lib/python3.11/site-packages/numpy/core/fromnumeric.py:59: FutureWarning: 'DataFrame.swapaxes' is deprecated and will be removed in a future version. Please use 'DataFrame.transpose' instead.
  return bound(*args, **kwds)
chunks: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 7/7 [00:03<00:00,  1.87it/s]


## Misc. tests

In [39]:
df_test: pd.DataFrame = gds.run_cypher(
  """
MATCH (ent:Entity)-[:RESOLVES]->(org:Organization)
RETURN ent.uid, ent.name
LIMIT 5
  """
)

df_test

,ent.uid,ent.name
0,1,C H. PETT WILL TRUST - RESIDUARY FUND
1,10,IRANJA FUND
2,100,ELLIS BIRK YOUTH TRUST
3,1000,EVELYN PARTNERS INVESTMENT MANAGEMENT SERVICES...
4,10000,COOPER'S HILL FUNDING PLC


In [40]:
df_test: pd.DataFrame = gds.run_cypher(
  """
MATCH (ent1:Entity)-[:RELATED]->(ent2:Entity)
RETURN ent1.name, ent2.name
LIMIT 5
  """
)

df_test

,ent1.name,ent2.name
0,C H. PETT WILL TRUST - RESIDUARY FUND,J T. CLARKE WILL TRUST - LATE E T CLARKE FOUR ...
1,PARITY BIDCO LIMITED,PIRUM SYSTEMS LIMITED
2,PARITY BIDCO LIMITED,WIC (UK) LTD
3,PARITY BIDCO LIMITED,NUCLEUS BIDCO LIMITED
4,PARITY BIDCO LIMITED,ADEXA LIMITED


In [41]:
df_test: pd.DataFrame = gds.run_cypher(
  """
MATCH (org:Organization)
RETURN org.record_id, org.addresses
LIMIT 5
  """
)

df_test

,org.record_id,org.addresses
0,NK-276gB7L8sFEhctkuEpZsus,"[{'ADDR_FULL': 'PO Box 15175/584, 6th Floor, S..."
1,NK-2RoX4fFRx9FtQE9osMzwk9,[{'ADDR_FULL': 'Сполучене Королівство Великої ...
2,NK-2WqqzAmKXFyrgf7xMDjYMv,"[{'ADDR_FULL': 'LONDON, ONTARIO, CANADA, TX'},..."
3,NK-2uLiev7s4AV8Rkw2bhtbGj,[{'ADDR_FULL': 'CHISWICK 52 JERMYN STREET LOND...
4,NK-3KRoVfPHitLoVonaD9y6bE,"[{'ADDR_FULL': '1 College Yard, Winchester Ave..."


In [42]:
"lei-213800HI8LJWKODMEM62" in os_id_set

True

In [43]:
df_test: pd.DataFrame = gds.run_cypher(
  """
MATCH (org:Organization)
WHERE org.record_id CONTAINS 'lei-213800HI8LJWKODMEM62'
RETURN org.record_id, org.addresses
LIMIT 5
  """
)

df_test

,org.record_id,org.addresses
0,lei-213800HI8LJWKODMEM62,[{'ADDR_FULL': 'C/O APEX CORPORATE TRUSTEES (U...
